# Insurance Fraud Detection — Data Preprocessing

Goal: clean and transform `insurance_claims.csv` into a model-ready feature matrix.

In [ ]:
# https://www.kaggle.com/code/niteshyadav3103/insurance-fraud-detection-using-12-models/notebook

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Load & Initial Inspection

In [3]:
df = pd.read_csv('insurance_claims.csv')
print(f'Shape: {df.shape}')
df.head(3)

Shape: (1000, 40)


,months_as_customer,age,policy_number,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,insured_sex,insured_education_level,insured_occupation,insured_hobbies,insured_relationship,capital-gains,capital-loss,incident_date,incident_type,collision_type,incident_severity,authorities_contacted,incident_state,incident_city,incident_location,incident_hour_of_the_day,number_of_vehicles_involved,property_damage,bodily_injuries,witnesses,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported,_c39
0,328,48,521585,2014-10-17,OH,250/500,1000,1406.91,0,466132,MALE,MD,craft-repair,sleeping,husband,53300,0,2015-01-25,Single Vehicle Collision,Side Collision,Major Damage,Police,SC,Columbus,9935 4th Drive,5,1,YES,1,2,YES,71610,6510,13020,52080,Saab,92x,2004,Y,NaN
1,228,42,342868,2006-06-27,IN,250/500,2000,1197.22,5000000,468176,MALE,MD,machine-op-inspct,reading,other-relative,0,0,2015-01-21,Vehicle Theft,?,Minor Damage,Police,VA,Riverwood,6608 MLK Hwy,8,1,?,0,0,?,5070,780,780,3510,Mercedes,E400,2007,Y,NaN
2,134,29,687698,2000-09-06,OH,100/300,2000,1413.14,5000000,430632,FEMALE,PhD,sales,board-games,own-child,35100,0,2015-02-22,Multi-vehicle Collision,Rear Collision,Minor Damage,Police,NY,Columbus,7121 Francis Lane,7,3,NO,2,3,NO,34650,7700,3850,23100,Dodge,RAM,2007,N,NaN


In [4]:
df.dtypes

months_as_customer               int64
age                              int64
policy_number                    int64
policy_bind_date                object
policy_state                    object
policy_csl                      object
policy_deductable                int64
policy_annual_premium          float64
umbrella_limit                   int64
insured_zip                      int64
insured_sex                     object
insured_education_level         object
insured_occupation              object
insured_hobbies                 object
insured_relationship            object
capital-gains                    int64
capital-loss                     int64
incident_date                   object
incident_type                   object
collision_type                  object
incident_severity               object
authorities_contacted           object
incident_state                  object
incident_city                   object
incident_location               object
incident_hour_of_the_day 

In [5]:
df.describe(include='all')

,months_as_customer,age,policy_number,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,insured_sex,insured_education_level,insured_occupation,insured_hobbies,insured_relationship,capital-gains,capital-loss,incident_date,incident_type,collision_type,incident_severity,authorities_contacted,incident_state,incident_city,incident_location,incident_hour_of_the_day,number_of_vehicles_involved,property_damage,bodily_injuries,witnesses,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported,_c39
count,1000.00,1000.00,1000.00,1000,1000,1000,1000.00,1000.00,1000.00,1000.00,1000,1000,1000,1000,1000,1000.00,1000.00,1000,1000,1000,1000,909,1000,1000,1000,1000.00,1000.00,1000,1000.00,1000.00,1000,1000.00,1000.00,1000.00,1000.00,1000,1000,1000.00,1000,0.00
unique,NaN,NaN,NaN,951,3,3,NaN,NaN,NaN,NaN,2,7,14,20,6,NaN,NaN,60,4,4,4,4,7,7,1000,NaN,NaN,3,NaN,NaN,3,NaN,NaN,NaN,NaN,14,39,NaN,2,NaN
top,NaN,NaN,NaN,2006-01-01,OH,250/500,NaN,NaN,NaN,NaN,FEMALE,JD,machine-op-inspct,reading,own-child,NaN,NaN,2015-02-02,Multi-vehicle Collision,Rear Collision,Minor Damage,Police,NY,Springfield,9935 4th Drive,NaN,NaN,?,NaN,NaN,?,NaN,NaN,NaN,NaN,Saab,RAM,NaN,N,NaN
freq,NaN,NaN,NaN,3,352,351,NaN,NaN,NaN,NaN,537,161,93,64,183,NaN,NaN,28,419,292,354,292,262,157,1,NaN,NaN,360,NaN,NaN,343,NaN,NaN,NaN,NaN,80,43,NaN,753,NaN
mean,203.95,38.95,546238.65,NaN,NaN,NaN,1136.00,1256.41,1101000.00,501214.49,NaN,NaN,NaN,NaN,NaN,25126.10,-26793.70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.64,1.84,NaN,0.99,1.49,NaN,52761.94,7433.42,7399.57,37928.95,NaN,NaN,2005.10,NaN,NaN
std,115.11,9.14,257063.01,NaN,NaN,NaN,611.86,244.17,2297406.60,71701.61,NaN,NaN,NaN,NaN,NaN,27872.19,28104.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.95,1.02,NaN,0.82,1.11,NaN,26401.53,4880.95,4824.73,18886.25,NaN,NaN,6.02,NaN,NaN
min,0.00,19.00,100804.00,NaN,NaN,NaN,500.00,433.33,-1000000.00,430104.00,NaN,NaN,NaN,NaN,NaN,0.00,-111100.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,1.00,NaN,0.00,0.00,NaN,100.00,0.00,0.00,70.00,NaN,NaN,1995.00,NaN,NaN
25%,115.75,32.00,335980.25,NaN,NaN,NaN,500.00,1089.61,0.00,448404.50,NaN,NaN,NaN,NaN,NaN,0.00,-51500.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.00,1.00,NaN,0.00,1.00,NaN,41812.50,4295.00,4445.00,30292.50,NaN,NaN,2000.00,NaN,NaN
50%,199.50,38.00,533135.00,NaN,NaN,NaN,1000.00,1257.20,0.00,466445.50,NaN,NaN,NaN,NaN,NaN,0.00,-23250.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.00,1.00,NaN,1.00,1.00,NaN,58055.00,6775.00,6750.00,42100.00,NaN,NaN,2005.00,NaN,NaN
75%,276.25,44.00,759099.75,NaN,NaN,NaN,2000.00,1415.70,0.00,603251.00,NaN,NaN,NaN,NaN,NaN,51025.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.00,3.00,NaN,2.00,2.00,NaN,70592.50,11305.00,10885.00,50822.50,NaN,NaN,2010.00,NaN,NaN


## 2. Replace `?` Placeholders with NaN

In [ ]:
df.replace('?', np.nan, inplace=True)

missing = df.isnull().sum()
missing[missing > 0]

## 3. Drop Irrelevant / Leaky Columns

| Column | Reason |
|---|---|
| `_c39` | Entirely empty artifact column |
| `policy_number` | Unique row ID, no predictive signal |
| `insured_zip` | Near-unique identifier (995/1000 distinct) |
| `incident_location` | Unique free-text address |
| `policy_bind_date` | Replaced by engineered `policy_tenure_years` |
| `incident_date` | Replaced by engineered `incident_month` |

In [ ]:
# --- Feature engineering from dates before dropping ---

# policy tenure in years at time of incident
df['policy_bind_date'] = pd.to_datetime(df['policy_bind_date'], errors='coerce')
df['incident_date']    = pd.to_datetime(df['incident_date'],    errors='coerce')
df['policy_tenure_years'] = ((df['incident_date'] - df['policy_bind_date']).dt.days / 365.25).round(1)

# month of incident (seasonality signal)
df['incident_month'] = df['incident_date'].dt.month

COLS_TO_DROP = [
    '_c39',
    'policy_number',
    'insured_zip',
    'incident_location',
    'policy_bind_date',
    'incident_date',
]
df.drop(columns=COLS_TO_DROP, inplace=True)
print(f'Shape after drop: {df.shape}')

## 4. Handle Missing Values

- `collision_type` (178 missing): mode imputation — most incidents have a known collision type
- `property_damage` (360 missing): mode imputation (binary YES/NO)
- `police_report_available` (343 missing): mode imputation (binary YES/NO)

In [ ]:
for col in ['collision_type', 'property_damage', 'police_report_available']:
    mode_val = df[col].mode()[0]
    df[col].fillna(mode_val, inplace=True)
    print(f'{col}: filled with mode = "{mode_val}"')

# confirm no nulls remain
assert df.isnull().sum().sum() == 0, 'Remaining nulls!'
print('\nNo missing values remaining.')

## 5. Encode Target Variable

In [ ]:
df['fraud_reported'] = (df['fraud_reported'] == 'Y').astype(int)
print(df['fraud_reported'].value_counts())

fraud_rate = df['fraud_reported'].mean()
print(f'\nFraud rate: {fraud_rate:.1%}')

## 6. Encode Categorical Features

**Binary (YES/NO, MALE/FEMALE)** → 0/1  
**Ordinal** → integer order  
**Nominal multi-class** → `pd.get_dummies` (one-hot, `drop_first=True` to avoid multicollinearity)

In [ ]:
# --- Binary encodings ---
df['insured_sex']           = (df['insured_sex'] == 'MALE').astype(int)
df['property_damage']       = (df['property_damage'] == 'YES').astype(int)
df['police_report_available'] = (df['police_report_available'] == 'YES').astype(int)

# --- Ordinal: education level ---
edu_order = {'JD': 0, 'High School': 1, 'Associate': 2, 'College': 3, 'Bachelor': 4, 'Masters': 5, 'PhD': 6}
df['insured_education_level'] = df['insured_education_level'].map(edu_order)

# --- Ordinal: incident_severity ---
severity_order = {'Trivial Damage': 0, 'Minor Damage': 1, 'Major Damage': 2, 'Total Loss': 3}
df['incident_severity'] = df['incident_severity'].map(severity_order)

# --- Ordinal: policy_csl (combined single limit) ---
csl_order = {'100/300': 0, '250/500': 1, '500/1000': 2}
df['policy_csl'] = df['policy_csl'].map(csl_order)

print('Binary/ordinal encodings done.')

In [ ]:
# --- One-hot encode remaining nominal categoricals ---
nominal_cols = [
    'policy_state',
    'insured_occupation',
    'insured_hobbies',
    'insured_relationship',
    'incident_type',
    'collision_type',
    'authorities_contacted',
    'incident_state',
    'incident_city',
    'auto_make',
    'auto_model',
]

df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)
print(f'Shape after one-hot encoding: {df.shape}')

## 7. Correlation Check (top features vs target)

In [ ]:
corr = df.corr()['fraud_reported'].drop('fraud_reported').abs().sort_values(ascending=False)

plt.figure(figsize=(8, 6))
corr.head(20).plot(kind='barh', color='steelblue')
plt.title('Top 20 Features by |Correlation| with fraud_reported')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(corr.head(10).to_string())

## 8. Train / Test Split

In [ ]:
X = df.drop(columns=['fraud_reported'])
y = df['fraud_reported']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train fraud rate: {y_train.mean():.1%}  |  Test fraud rate: {y_test.mean():.1%}')

## 9. Scale Numeric Features

Fit `StandardScaler` on **train only**, then transform both splits to prevent data leakage.

In [ ]:
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])

print(f'Scaled {len(numeric_cols)} numeric columns.')
X_train.describe().loc[['mean','std']].round(2)

## 10. Save Preprocessed Data

In [ ]:
X_train.assign(fraud_reported=y_train.values).to_csv('train.csv', index=False)
X_test.assign(fraud_reported=y_test.values).to_csv('test.csv', index=False)

print('Saved: train.csv and test.csv')
print(f'Final feature count: {X_train.shape[1]}')